# Event Catering Preparation Under Demand Uncertainty
### Monte Carlo + next-event discrete-event simulation of how much food to prepare

**Summary.** Caterers prepare food from advance bookings, but real attendance and per-guest consumption are random. Too little food causes shortages; too much wastes money. This notebook models attendance (Binomial), per-guest consumption (discrete distribution) and the buffet itself (next-event queueing simulation with inventory depletion), then runs **10,000 Monte Carlo replications per scenario** to find the preparation quantity $Q$ that minimises expected cost subject to a **shortage probability of at most 5%**. The simulator is verified against analytic results (Binomial moments, M/M/1 and Erlang-C queue waits, a hand-solved optimisation case, the newsvendor quantile), and a sensitivity study, a $N^{-1/2}$ convergence study and a multi-category (rice / meat / vegetables) extension are included.

> **All cost, consumption, arrival and service parameters are ASSUMED** (no measured catering records were used). Sensitivity analysis in §5 shows how much each assumption matters. Random seed: `SEED = 42` throughout.

In [1]:
# Setup (Colab: packages are preinstalled; this keeps the notebook reproducible elsewhere)
!pip install -q numpy scipy matplotlib pandas

import os, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from math import factorial

SEED = 42                                   # master seed (reproducibility)
os.makedirs("figures", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "font.size": 10})
T0 = time.time()

## 1. Problem and question

Event caterers in Sri Lanka typically cook against advance guest bookings. On the day, the number who attend and how many portions each guest takes are uncertain. Shortages hurt customer satisfaction; surplus wastes ingredients (the food-service sector is a major source of food waste [5]).

**Question.** For an event with $N$ invited guests (100 to 500), *how many portions $Q$ should be prepared to minimise expected total cost while keeping the probability of any food shortage at or below 5%?* And how sensitive is that answer to attendance rate, consumption variability and the cost structure?

## 2. Model, assumptions and equations

**Assumptions** (all parameters assumed; see §5 sensitivity)
1. Each invited guest attends independently with probability $p$.
2. Each attending guest takes $X$ portions, i.i.d., from a discrete distribution on $\{0,1,2,3\}$ (mean 1.5).
3. Guests arrive as a Poisson stream (rate $\lambda$) truncated after the $A$ attendees; $c$ servers at the buffet, exponential service (rate $\mu$ each), FIFO queue.
4. Food is taken when a guest's service completes; if stock runs short the guest receives what remains (partial service) and the rest is *unmet demand*.
5. Preparation is fixed before the event (single period, no re-cooking). Costs: $c_\text{prep}$ per prepared portion, $c_\text{short}$ per unmet portion (LKR).

**Input modelling and random variate generation**

$$A\sim\text{Binomial}(N,p),\qquad E[A]=Np,\qquad D=\sum_{i=1}^{A}X_i .$$

Given $A$, the numbers of guests taking each portion size are $\text{Multinomial}(A,\boldsymbol\pi)$, so $D=\sum_k v_k n_k$ is generated in one vectorised draw. By the law of total variance
$$E[D]=Np\,\mu_X,\qquad \operatorname{Var}(D)=Np\,\sigma_X^2+Np(1-p)\,\mu_X^2 .$$
Inter-arrival times use inverse-transform sampling: $T=-\ln(U)/\lambda$, $U\sim U[0,1]$.

**Cost and decision problem**
$$C(Q)=c_\text{prep}\,Q+c_\text{short}\,(D-Q)^+,\qquad Q^\*=\arg\min_Q E[C(Q)]\ \ \text{s.t.}\ \ P(D>Q)\le\alpha=0.05 .$$
Unconstrained, a one-unit increase in $Q$ changes expected cost by $c_\text{prep}-c_\text{short}P(D>Q)$, so the optimum is the smallest $Q$ with
$$F_D(Q)\ \ge\ 1-\frac{c_\text{prep}}{c_\text{short}}\quad(\text{newsvendor critical fractile}).$$

**Monte Carlo estimation (law of large numbers)**
$$\hat C=\frac1n\sum_{j=1}^{n}C_j,\qquad \text{95\% CI}=\hat C\pm1.96\,\frac{s}{\sqrt n},\qquad \text{error}\propto n^{-1/2}.$$
All candidate $Q$ are evaluated on the *same* $n$ demand draws (common random numbers), which removes noise from comparisons between quantities.

**Discrete-event simulation (next-event time advance).** State: clock, busy servers, FIFO queue, inventory. Future-event list (binary heap) holds ARRIVAL and DEPARTURE events; the clock jumps to the earliest event. Ties are broken deterministically (departures before arrivals, then insertion order). For queue verification with $\rho=\lambda/(c\mu)<1$:
$$W_q^{M/M/1}=\frac{\lambda}{\mu(\mu-\lambda)},\qquad W_q^{M/M/c}=\frac{C(c,a)}{c\mu-\lambda},\quad C(c,a)=\frac{\frac{a^c}{c!}\frac1{1-\rho}}{\sum_{k=0}^{c-1}\frac{a^k}{k!}+\frac{a^c}{c!}\frac1{1-\rho}},\ a=\frac\lambda\mu .$$

| Component | Concept applied |
|---|---|
| $A$, $X$, inter-arrival and service draws | Input modelling and random variate generation |
| $\hat C$, CIs, shortage probability, $n^{-1/2}$ study | Monte Carlo integration and the law of large numbers |
| Buffet queue, event list, inventory depletion, M/M/c checks | Discrete-event simulation and queueing |

## 3. Implementation

The model is split into: parameters and input models, a fast vectorised Monte Carlo layer (used for the grid search), the next-event DES layer, and a multi-category extension.

In [2]:
# 3.0  Imports and conventions
from dataclasses import dataclass, field, replace
import heapq
import numpy as np

print('Model code loaded')

Model code loaded


In [3]:
# 3.  Parameters
@dataclass(frozen=True)
class Params:
    N: int = 200                      # invited guests
    p: float = 0.90                   # P(an invited guest attends)
    values: tuple = (0, 1, 2, 3)      # portions a guest may take
    probs: tuple = (0.05, 0.50, 0.35, 0.10)   # their probabilities (mean 1.5)
    c_prep: float = 300.0             # LKR per prepared portion
    c_short: float = 1500.0           # LKR penalty per unmet portion
    lam: float = 5.0                  # guest arrival rate (guests/min)
    mu: float = 1.5                   # service rate per server (guests/min)
    servers: int = 4                  # buffet service points

    @property
    def mean_portions(self):
        return float(np.dot(self.values, self.probs))

    @property
    def var_portions(self):
        v = np.array(self.values, float)
        return float(np.dot(v**2, self.probs) - self.mean_portions**2)

    @property
    def mean_demand(self):            # E[D] = N p E[X]  (Wald)
        return self.N * self.p * self.mean_portions

    @property
    def var_demand(self):             # Var D = Np E[X^2] ... via law of total variance
        EX, VX = self.mean_portions, self.var_portions
        return self.N * self.p * VX + self.N * self.p * (1 - self.p) * EX**2


In [4]:
# 3.  Input models (random variate generation)
def draw_attendance(par, rng, size):
    """A ~ Binomial(N, p)."""
    return rng.binomial(par.N, par.p, size=size)


def draw_demand(par, rng, size):
    """Total portions D = sum of X_i over the A attending guests.
    Given A, the counts of guests choosing each portion size are
    Multinomial(A, probs); D = counts . values.  Fully vectorised."""
    A = draw_attendance(par, rng, size)
    counts = rng.multinomial(A, par.probs)          # shape (size, len(values))
    return A, counts @ np.array(par.values)


In [5]:
# 3.  Fast layer: Monte Carlo + grid search with common random numbers
def ci95(x, axis=0):
    """Mean and 95% half-width (normal approx, 1.96 s/sqrt(n))."""
    n = x.shape[axis]
    m = x.mean(axis=axis)
    hw = 1.96 * x.std(axis=axis, ddof=1) / np.sqrt(n)
    return m, hw


def evaluate_grid(par, Qgrid, reps=10_000, seed=42):
    """Evaluate every Q in Qgrid on the SAME `reps` demand draws."""
    rng = np.random.default_rng(seed)
    _, D = draw_demand(par, rng, reps)
    Q = np.asarray(Qgrid)[None, :]
    D = D[:, None]
    unmet = np.maximum(D - Q, 0)
    left = np.maximum(Q - D, 0)
    short = (unmet > 0).astype(float)
    cost = par.c_prep * Q + par.c_short * unmet
    out = {"Q": np.asarray(Qgrid)}
    for name, arr in [("cost", cost), ("unused", left), ("unmet", unmet), ("p_short", short)]:
        m, hw = ci95(arr)
        out[name], out[name + "_hw"] = m, hw
    return out


def optimise(par, Qgrid, reps=10_000, seed=42, alpha=0.05):
    """Min expected cost s.t. P(shortage) <= alpha.  Returns dict."""
    g = evaluate_grid(par, Qgrid, reps, seed)
    feas = g["p_short"] <= alpha
    if not feas.any():
        raise ValueError("No Q in grid satisfies the shortage constraint; widen the grid.")
    idx_f = np.where(feas)[0]
    i = idx_f[np.argmin(g["cost"][idx_f])]
    j = int(np.argmin(g["cost"]))                    # unconstrained optimum
    return {"grid": g, "i_opt": i, "Q_opt": int(g["Q"][i]), "i_unc": j,
            "Q_unc": int(g["Q"][j]), "constraint_binding": bool(i != j)}
